In [ ]:
# function to save matchuptable


from openpyxl import Workbook
from openpyxl.styles import Alignment

def generate_matchup_excel(MatchupTable_OpenDrive, output_filename="MatchupTable_OpenDrive.xlsx"):

    network_columns = ["JunctionID_OpenDrive", "Bearing", "Numbering", "FromRoadID_OpenDrive", "ToRoadID_OpenDrive", "Turn"]
    demand_columns = ["File_GridSmart", "Date_GridSmart", "IntersectionName_GridSmart", "Turn_GridSmart"]
    signal_columns = ["File_Synchro", "IntersectionID_Synchro", "Turn_Synchro"]
    other_columns = ["Need calibration?"]
    
    wb = Workbook()
    ws = wb.active
    
    ws.append(["Network"] * len(network_columns) + ["Demand"] * len(demand_columns) +
              ["Signal"] * len(signal_columns) + [""] * len(other_columns))
    
    ws.merge_cells(start_row=1, start_column=1, end_row=1, end_column=len(network_columns))
    ws.merge_cells(start_row=1, start_column=len(network_columns) + 1, end_row=1,
                   end_column=len(network_columns) + len(demand_columns))
    ws.merge_cells(start_row=1, start_column=len(network_columns) + len(demand_columns) + 1, end_row=1,
                   end_column=len(network_columns) + len(demand_columns) + len(signal_columns))
    
    ws.append(network_columns + demand_columns + signal_columns + other_columns)

    for row in MatchupTable_OpenDrive.itertuples(index=False):
        ws.append(list(row))
    
    current_start = 3  # Data starts at row 3
    for i in range(3, len(MatchupTable_OpenDrive) + 3):
        if (i == len(MatchupTable_OpenDrive) + 2 or
                ws[f"A{i}"].value != ws[f"A{i+1}"].value):  # Check next row
            if current_start < i:  # Only merge if there are multiple same values
                ws.merge_cells(start_row=current_start, start_column=1, end_row=i, end_column=1)  # JunctionID_OpenDrive
                ws.merge_cells(start_row=current_start, start_column=7, end_row=i, end_column=7)  # File_GridSmart
                ws.merge_cells(start_row=current_start, start_column=8, end_row=i, end_column=8)  # Date_GridSmart
                ws.merge_cells(start_row=current_start, start_column=9, end_row=i, end_column=9)  # IntersectionName_GridSmart
                # ws.merge_cells(start_row=current_start, start_column=11, end_row=i, end_column=11)  # File_Synchro
                ws.merge_cells(start_row=current_start, start_column=12, end_row=i, end_column=12)  # IntersectionID_Synchro
                ws.merge_cells(start_row=current_start, start_column=14, end_row=i, end_column=14)  # Need calibration?
            current_start = i + 1
            
    if len(MatchupTable_OpenDrive) > 0:
        ws.merge_cells(start_row=3, start_column=11, end_row=len(MatchupTable_OpenDrive) + 2, end_column=11)
  
    # Center align merged cells
    for row in ws.iter_rows():
        for cell in row:
            cell.alignment = Alignment(horizontal="center", vertical="center")
    
    # Adjust column widths
    column_widths = {
        "A": 20,  # JunctionID_OpenDrive
        "B": 15,  # Bearing
        "C": 15,  # Numbering
        "D": 25,  # FromRoadID_OpenDrive
        "E": 25,  # ToRoadID_OpenDrive
        "F": 15,  # Turn
        "G": 20,  # File_GridSmart
        "H": 20,  # Date_GridSmart
        "I": 30,  # IntersectionName_GridSmart
        "J": 20,  # Turn_GridSmart
        "K": 20,  # File_Synchro
        "L": 25,  # IntersectionID_Synchro
        "M": 20,  # Turn_Synchro
        "N": 20   # Need calibration?
    }
    for col, width in column_widths.items():
        ws.column_dimensions[col].width = width
    
    wb.save(output_filename)



# generate_matchup_excel(MatchupTable_OpenDrive, "MatchupTable_OpenDrive.xlsx")

In [ ]:
# function to read matchuptable
import pandas as pd
matchup_file_path = "MatchupTable.xlsx"
MatchupTable_UserInput = pd.read_excel(matchup_file_path, skiprows=1, dtype=str)
merged_columns = ["JunctionID_OpenDrive","File_GridSmart","Date_GridSmart","IntersectionName_GridSmart","File_Synchro","IntersectionID_Synchro","Need calibration?"]
MatchupTable_UserInput[merged_columns] = MatchupTable_UserInput[merged_columns].ffill()
